In [3]:
import os
import numpy as np
import pandas as pd

# ==============================
# Paths
# ==============================
base_path = "../../../data/simulation/"
X_out_path = os.path.join(base_path, "engine_total_X.npy")
y_out_path = os.path.join(base_path, "engine_total_y.npy")
csv_out_path = os.path.join(base_path, "engine_total.csv")
os.makedirs(base_path, exist_ok=True)

# ==============================
# Load data
# ==============================
engine_start_X = np.load(os.path.join(base_path, "engine_start_X.npy"), allow_pickle=True)
engine_start_y = np.load(os.path.join(base_path, "engine_start_y.npy"), allow_pickle=True)

engine_off_X = np.load(os.path.join(base_path, "engine_off_X.npy"), allow_pickle=True)
engine_off_y = np.load(os.path.join(base_path, "engine_off_y.npy"), allow_pickle=True)

engine_OCC_X = np.load(os.path.join(base_path, "engine_occ_X.npy"), allow_pickle=True)
engine_OCC_y = np.load(os.path.join(base_path, "engine_occ_y.npy"), allow_pickle=True)

engine_normal_X = np.load(os.path.join(base_path, "engine_normal_load_X.npy"), allow_pickle=True)
engine_normal_y = np.load(os.path.join(base_path, "engine_normal_load_y.npy"), allow_pickle=True)

engine_high_X = np.load(os.path.join(base_path, "engine_high_load_X.npy"), allow_pickle=True)
engine_high_y = np.load(os.path.join(base_path, "engine_high_load_y.npy"), allow_pickle=True)

engine_critical_X = np.load(os.path.join(base_path, "engine_critical_load_X.npy"), allow_pickle=True)
engine_critical_y = np.load(os.path.join(base_path, "engine_critical_load_y.npy"), allow_pickle=True)

# ==============================
# Helpers (use REAL y only; no auto-labeling / no mapping)
# ==============================
def expand_labels_to_T_str(y_seq, T):
    """
    Expand the PROVIDED y to length T WITHOUT altering content:
      - scalar/string/number  -> broadcast that exact value to length T
      - (T,)                  -> use as-is
      - (T,1)                 -> flatten and use
    """
    if not hasattr(y_seq, "shape"):
        return np.array([y_seq] * T, dtype=object)

    y_arr = np.asarray(y_seq, dtype=object)

    # scalar-like ndarray -> broadcast
    if y_arr.ndim == 0 or (y_arr.ndim == 1 and y_arr.size == 1):
        val = (y_arr.item() if y_arr.size else "")
        return np.array([val] * T, dtype=object)

    # (T,) -> as-is
    if y_arr.ndim == 1 and y_arr.size == T:
        return y_arr.astype(object)

    # (T,1) -> flatten -> as-is
    if y_arr.ndim == 2 and y_arr.shape == (T, 1):
        return y_arr.reshape(-1).astype(object)

    # Let it error loudly if shape is unexpected (no "fixing")
    raise ValueError(f"Unsupported y shape {y_arr.shape}; expected scalar, (T,), or (T,1)")

def representative_label(y_exp: np.ndarray) -> str:
    """
    Use the center timestep label; if it's 'Unknown', fallback to the first
    non-'Unknown'. If none found, return 'Unknown'.
    """
    if y_exp is None or len(y_exp) == 0:
        return "Unknown"
    center = y_exp[len(y_exp)//2]
    if center != "Unknown":
        return center
    for v in y_exp:
        if v != "Unknown":
            return v
    return "Unknown"

def reorder_sequences_with_unknown(seq_labels, OFF_SET, START_SET, NORMAL_SET, HIGH_SET, CRIT_SET, UNKNOWN_SET, unk_per_cycle=5, probs=(0.6, 0.2, 0.2), seed=42):
    """
    Reorder into cycles: OFF -> START -> NORMAL -> HIGH -> CRITICAL, with an
    Unknown block inserted per cycle according to probs:
      - 60%: between loads (between Normal/High and/or High/Critical)
      - 20%: between START and first LOAD
      - 20%: between OFF and START
    Never place Unknown before OFF or between the end of LOAD and next OFF.

    All matching is by exact label strings already present (no remapping).
    """
    from collections import defaultdict, deque
    rng = np.random.default_rng(seed)

    pools = defaultdict(deque)
    for i, lbl in enumerate(seq_labels):
        pools[lbl].append(i)

    def pop_one(label_list):
        for lbl in label_list:
            dq = pools.get(lbl)
            if dq and len(dq) > 0:
                return dq.popleft()
        return None

    def pop_unknown(k):
        taken = []
        if not UNKNOWN_SET:
            return taken
        for _ in range(k):
            got = pop_one(UNKNOWN_SET)
            if got is None:
                break
            taken.append(got)
        return taken

    OFF_SET = list(OFF_SET)
    START_SET = list(START_SET)
    NORMAL_SET = list(NORMAL_SET)
    HIGH_SET = list(HIGH_SET)
    CRIT_SET = list(CRIT_SET)

    ordered = []
    N = len(seq_labels)

    while True:
        o = pop_one(OFF_SET)
        s = pop_one(START_SET)
        n = pop_one(NORMAL_SET)
        h = pop_one(HIGH_SET)
        c = pop_one(CRIT_SET)
        if None in (o, s, n, h, c):
            break

        where = rng.choice(["between_loads", "start_load", "off_start"], p=np.array(probs, dtype=float))
        unk = pop_unknown(unk_per_cycle)

        if where == "between_loads":
            split = rng.integers(0, len(unk) + 1) if len(unk) > 0 else 0
            u1, u2 = unk[:split], unk[split:]
            block = [o, s, n] + u1 + [h] + u2 + [c]
        elif where == "start_load":
            block = [o, s] + unk + [n, h, c]
        else:  # off_start
            block = [o] + unk + [s, n, h, c]

        ordered.extend(block)

    used = set(ordered)
    leftovers = [i for i in range(N) if i not in used]
    # CHANGED: group leftovers by label so cold/warm/start sequences are not
    # interleaved at the tail of the dataset (kills the nonsensical thermal
    # jumps like cold->warm->cold->warm at the end).
    leftovers.sort(key=lambda i: seq_labels[i])
    ordered.extend(leftovers)
    return ordered

# ==============================
# Build ragged X_total, y_total (REAL labels), reorder by pattern, and save CSV
# ==============================
groups = [
    ("Start",         engine_start_X,    engine_start_y,    "Engine Start"),
    ("Off",           engine_off_X,      engine_off_y,      "Engine Off (cold)"),
    ("OCC",           engine_OCC_X,      engine_OCC_y,      "Unknown"),
    ("NormalLoad",    engine_normal_X,   engine_normal_y,   "Normal Load (idle)"),
    ("HighLoad",      engine_high_X,     engine_high_y,     "High Load (idle)"),
    ("CriticalLoad",  engine_critical_X, engine_critical_y, "Critical Load (idle)"),
]

def _to_numeric_or_object(X_seq):
    """
    Try to cast to float32 for performance. If any value is non-numeric (gibberish),
    fall back to dtype=object to preserve the original values.
    """
    try:
        return np.asarray(X_seq, dtype=np.float32)
    except Exception:
        # Mixed types -> keep as object without altering content
        arr = np.asarray(X_seq, dtype=object)
        # Optional: leave as-is; do not attempt coercion
        return arr

def _csv_val(v):
    """Keep strings as-is; cast numeric-like to float (for clean CSV)."""
    if isinstance(v, (np.floating, float, int, np.integer)):
        return float(v)
    try:
        return float(v)  # numeric in string form
    except Exception:
        return str(v)

# 1) Collect sequences and their representative labels (exact strings; no mapping)
seq_X, seq_Yorig, seq_Yexp, seq_label = [], [], [], []
for _, Xg, Yg, _ in groups:
    for i, X_seq in enumerate(Xg):
        X_seq = _to_numeric_or_object(X_seq)  # <- tolerant to gibberish
        T = int(X_seq.shape[0])

        if hasattr(Yg, "__len__") and not isinstance(Yg, (str, bytes)) and len(Yg) > i:
            y_seq = Yg[i]
        else:
            y_seq = Yg

        y_orig = np.asarray(y_seq, dtype=object)
        y_exp  = expand_labels_to_T_str(y_seq, T)

        seq_X.append(X_seq)
        seq_Yorig.append(y_orig)
        seq_Yexp.append(y_exp.astype(object))
        seq_label.append(representative_label(y_exp))

# 2) Build dynamic buckets (exact strings present) and pattern
present = set(seq_label)
OFF_SET    = sorted([l for l in present if "Off"      in l])
START_SET  = sorted([l for l in present if "Start"    in l])
NORMAL_SET = sorted([l for l in present if "Normal"   in l])
HIGH_SET   = sorted([l for l in present if "High"     in l])
CRIT_SET   = sorted([l for l in present if "Critical" in l])
UNKNOWN_SET= sorted([l for l in present if l == "Unknown" or "Unknown" in l])

# 3) Determine order and emit in that order
order_idx = reorder_sequences_with_unknown(
    seq_label,
    OFF_SET, START_SET, NORMAL_SET, HIGH_SET, CRIT_SET, UNKNOWN_SET,
    unk_per_cycle=3, probs=(0.6, 0.2, 0.2), seed=42
)

X_total_list, y_total_list = [], []
rows = []
for new_seq_id, idx in enumerate(order_idx):
    X_seq = seq_X[idx]
    y_orig = seq_Yorig[idx]
    y_exp  = seq_Yexp[idx]

    X_total_list.append(X_seq)
    y_total_list.append(y_orig)

    T = int(X_seq.shape[0])
    for t in range(T):
        x_t = X_seq[t]
        rows.append([
            new_seq_id,
            t,
            _csv_val(x_t[0]),
            _csv_val(x_t[1]),
            _csv_val(x_t[2]),
            _csv_val(x_t[3]),
            y_exp[t]
        ])

# 4) Save ragged arrays and CSV
X_total = np.array(X_total_list, dtype=object)  # ragged sequences; may contain strings
y_total = np.array(y_total_list, dtype=object)
np.save(X_out_path, X_total, allow_pickle=True)
np.save(y_out_path, y_total, allow_pickle=True)

df = pd.DataFrame(rows, columns=["sample", "timestep", "Temperature", "Pressure", "RPM", "Vibration", "State"])
df.to_csv(csv_out_path, index=False)

print("DONE")

DONE
